In [1]:
#Cell 1 - Installs + imports
!pip install pyautogui numpy matplotlib idun-guardian-sdk
!pip install keyboard
import pyautogui
import numpy as np
import time
import random
import asyncio
import threading
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from collections import deque
from idun_guardian_sdk import GuardianClient

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\eumey\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\eumey\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Cell 2 — Connect to IDUN Guardian 3
client = GuardianClient(api_token="idun_eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJqdGkiOiI4ZjM3NjRlMy0yNjA0LTQ3ZmItYThiOS1mNGU2YTkxNjc5ZGIiLCJ1aWQiOiI3MzE0YThjMi0xMGExLTcwNjQtZmIyOC0yN2UxNDVkZjg5NzUiLCJkaWQiOiJDMC1BRS1GMC04QS0xQy1BQiIsImlhdCI6MTc4MTQ1OTkwOS4yMjgxMTV9.tUUBFi-eVBoazeznORMnsM5GGjjMvTdGgn76LU-rU0o")

await client.connect_device()

battery = await client.check_battery()
print(f"Battery: {battery}%")

if battery < 20:
    print("WARNING: Low battery — charge before use")

[INFO] 2026-08-08 00:03:13,837: Device address not provided, searching for the device...
[INFO] 2026-08-08 00:03:19,120: [BLE]: Selected address C0:AE:F0:8A:1C:AB



----- Available devices -----

Index | Name | Address
----------------------------
0     | IGE-8A1CAB | C0:AE:F0:8A:1C:AB
----------------------------

Battery: 93%


In [3]:
#Cell 3 ── Configuration ──────────────────────────────────────────
DEAD_ZONE = 0.05
SMOOTHING = 0.3
REFRACTORY = 0.5
QUALITY_GOOD = 50
QUALITY_USABLE = 30
BUFFER_SIZE = 200

# ── Live data state ───────────────────────────────────────────────
latest_gyro = {"x": 0, "y": 0}
latest_eeg = 0.0
latest_quality = 0.0
latest_clench_time = 0
last_processed_click = 0
prev_x, prev_y = 0, 0
cursor_active = True

eeg_buffer = deque([0] * BUFFER_SIZE, maxlen=BUFFER_SIZE)
gyro_buffer_x = deque([0.0] * 5, maxlen=5)
gyro_buffer_y = deque([0.0] * 5, maxlen=5)
gyro_bias = {"x": 0, "y": 0}

# ── Handlers ──────────────────────────────────────────────────────
def handle_live_insights(event):
    global latest_gyro, latest_eeg
    msg = event.message
    if "imu" in msg and len(msg["imu"]) > 0:
        samples = msg["imu"]
        latest_gyro["x"] = sum(s.get("gyro_x", 0) for s in samples) / len(samples)
        latest_gyro["y"] = sum(s.get("gyro_y", 0) for s in samples) / len(samples)
        gyro_buffer_x.append(latest_gyro["x"])
        gyro_buffer_y.append(latest_gyro["y"])
    if "raw_eeg" in msg and len(msg["raw_eeg"]) > 0:
        latest_eeg = msg["raw_eeg"][-1]["ch1"]
        eeg_buffer.append(latest_eeg)

def handle_predictions(event):
    global latest_quality, latest_clench_time
    msg = event.message
    if msg["predictionType"] == "QUALITY_SCORE":
        latest_quality = msg["result"]["quality_score"]
    elif msg["predictionType"] == "JAW_CLENCH":
        latest_clench_time = time.time()
        print("JAW CLENCH DETECTED")

# ── Quality gate ──────────────────────────────────────────────────
def evaluate_quality(score):
    global cursor_active
    if score >= QUALITY_GOOD:
        cursor_active = True
        return "GOOD"
    elif score >= QUALITY_USABLE:
        cursor_active = True
        return "USABLE"
    else:
        cursor_active = False
        return "POOR"

# ── Calibration ───────────────────────────────────────────────────
def calibrate(seconds=2):
    global gyro_bias
    print(f"Hold still for {seconds} seconds...")
    samples_x, samples_y = [], []
    start = time.time()
    while time.time() - start < seconds:
        samples_x.append(latest_gyro["x"])
        samples_y.append(latest_gyro["y"])
        time.sleep(0.05)
    gyro_bias["x"] = sum(samples_x) / len(samples_x)
    gyro_bias["y"] = sum(samples_y) / len(samples_y)
    print(f"Bias: x={gyro_bias['x']:.4f}, y={gyro_bias['y']:.4f}")

# ── Click detection ───────────────────────────────────────────────
def check_click():
    global last_processed_click
    now = time.time()
    if (now - latest_clench_time) < 0.5 and (now - last_processed_click) > REFRACTORY:
        last_processed_click = now
        return True
    return False

print("Pipeline ready")

Pipeline ready


In [ ]:
#Cell 4 Visualizer
fig, (ax_eeg, ax_imu, ax_quality) = plt.subplots(3, 1, figsize=(12, 8))
fig.suptitle('IDUN Guardian 3 — Live Monitor', fontsize=14)

# EEG
ax_eeg.set_title('Raw EEG')
ax_eeg.set_ylim(-100, 100)
ax_eeg.set_xlim(0, BUFFER_SIZE)
ax_eeg.set_ylabel('Amplitude (µV)')
line_eeg, = ax_eeg.plot([], [], color='cyan', linewidth=0.8)

# IMU
ax_imu.set_title('IMU — Head Movement')
ax_imu.set_ylim(-40, 40)
ax_imu.set_xlim(0, BUFFER_SIZE)
ax_imu.set_ylabel('Degrees/sec')
line_pitch, = ax_imu.plot([], [], color='lime', linewidth=0.8, label='Pitch (up/down)')
line_yaw, = ax_imu.plot([], [], color='orange', linewidth=0.8, label='Yaw (left/right)')
ax_imu.legend(loc='upper right')

# Quality
ax_quality.set_title('Signal Quality')
ax_quality.set_ylim(0, 100)
ax_quality.set_xlim(0, BUFFER_SIZE)
ax_quality.set_ylabel('Score')
ax_quality.axhline(y=50, color='green', linestyle='--', alpha=0.5, label='Good (50)')
ax_quality.axhline(y=30, color='yellow', linestyle='--', alpha=0.5, label='Usable (30)')
quality_buffer = deque([0] * BUFFER_SIZE, maxlen=BUFFER_SIZE)
line_quality, = ax_quality.plot([], [], color='white', linewidth=0.8)
ax_quality.legend(loc='upper right')

quality_text = ax_quality.text(0.02, 0.85, '', transform=ax_quality.transAxes, color='white')

plt.tight_layout()

def update(frame):
    pitch, yaw = get_imu()
    eeg_val = get_eeg()
    score = get_quality()
    status = evaluate_quality(score)

    eeg_buffer.append(eeg_val)
    pitch_buffer.append(pitch)
    yaw_buffer.append(yaw)
    quality_buffer.append(score)

    x = list(range(BUFFER_SIZE))
    line_eeg.set_data(x, list(eeg_buffer))
    line_pitch.set_data(x, list(pitch_buffer))
    line_yaw.set_data(x, list(yaw_buffer))
    line_quality.set_data(x, list(quality_buffer))

    color = 'green' if score >= QUALITY_GOOD else 'yellow' if score >= QUALITY_USABLE else 'red'
    quality_text.set_text(f"Status: {status}")
    quality_text.set_color(color)

    if detect_jaw_clench():
        ax_eeg.axvline(x=BUFFER_SIZE-1, color='red', alpha=0.5, linewidth=1)

    return line_eeg, line_pitch, line_yaw, line_quality, quality_text

print("Visualizer ready")

In [ ]:
# Cell 5 — start stream and confirm data is flowing
client.subscribe_live_insights(raw_eeg=True, filtered_eeg=False, imu=True, handler=handle_live_insights)
client.subscribe_realtime_predictions(quality_score=True, jaw_clench=True, handler=handle_predictions)

async def run_with_check():
    asyncio.ensure_future(client.start_recording(recording_timer=300))
    await asyncio.sleep(2)
    for i in range(3):
        print(f"gyro: {latest_gyro}, quality: {latest_quality:.1f}, eeg: {latest_eeg:.1f}")
        await asyncio.sleep(0.5)
    print("Stream confirmed live — now run Cell 6")

await run_with_check()

[INFO] 2026-08-08 00:04:52,405: [CLIENT]: Starting recording
[INFO] 2026-08-08 00:04:52,406: [CLIENT]: Recording timer: 300 seconds
[INFO] 2026-08-08 00:04:52,406: [CLIENT]: Ensuring hardware version is synced with Cloud
[INFO] 2026-08-08 00:04:53,111: [CLIENT]: Hardware version already exists in the Cloud


gyro: {'x': 0, 'y': 0}, quality: 0.0, eeg: 0.0
gyro: {'x': 0, 'y': 0}, quality: 0.0, eeg: 0.0
gyro: {'x': 0, 'y': 0}, quality: 0.0, eeg: 0.0
Stream confirmed live — now run Cell 6


JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED
JAW CLENCH DETECTED


[INFO] 2026-08-08 00:10:08,377: [CLIENT]: Recording Finished
[INFO] 2026-08-08 00:10:08,377: [CLIENT]: Recording ID 1786161908679


In [8]:
# Diagnostic cell
# Debug cell — async version
async def watch_eeg():
    print("Watching EEG for 10 seconds — clench hard a few times...")
    for i in range(100):
        if len(eeg_buffer) >= 20:
            recent = list(eeg_buffer)[-20:]
            baseline = np.std(recent)
            latest_dev = abs(latest_eeg - np.mean(recent))
            ratio = latest_dev / baseline if baseline > 0 else 0
            print(f"ratio: {ratio:.2f}  std: {baseline:.1f}  eeg: {latest_eeg:.1f}")
        else:
            print(f"buffer: {len(eeg_buffer)} samples")
        await asyncio.sleep(0.1)

await watch_eeg()

Watching EEG for 10 seconds — clench hard a few times...
ratio: 0.93  std: 49228.8  eeg: -31528.2
ratio: 0.67  std: 49065.0  eeg: -109988.9
ratio: 0.67  std: 49065.0  eeg: -109988.9
ratio: 0.64  std: 49374.5  eeg: -109186.0
ratio: 0.64  std: 49374.5  eeg: -109186.0
ratio: 0.56  std: 48991.8  eeg: -104326.6
ratio: 0.56  std: 48991.8  eeg: -104326.6
ratio: 0.77  std: 49089.4  eeg: -39297.8
ratio: 0.67  std: 49281.2  eeg: -110721.0
ratio: 0.47  std: 49008.1  eeg: -100093.0
ratio: 0.47  std: 49008.1  eeg: -100093.0
ratio: 0.61  std: 48974.6  eeg: -110550.0
ratio: 0.61  std: 48974.6  eeg: -110550.0
ratio: 1.13  std: 49519.5  eeg: -133754.9
ratio: 1.60  std: 49782.3  eeg: 3187.0
ratio: 0.72  std: 49295.2  eeg: -112892.4
ratio: 0.72  std: 49295.2  eeg: -112892.4
ratio: 1.63  std: 49647.2  eeg: 4477.9
ratio: 1.63  std: 49647.2  eeg: 4477.9
ratio: 0.73  std: 49176.0  eeg: -113377.0
ratio: 0.73  std: 49176.0  eeg: -113377.0
ratio: 1.66  std: 49590.7  eeg: 6011.6
ratio: 1.66  std: 49590.7  eeg: 6

In [ ]:
# Cell 6 — cursor control loop
try:
    stop_cursor.set()
    time.sleep(0.1)
except:
    pass

stop_cursor = threading.Event()
pyautogui.FAILSAFE = False

time.sleep(0.5)
bias_x = latest_gyro["x"]
bias_y = latest_gyro["y"]
print(f"Bias captured: x={bias_x:.4f}, y={bias_y:.4f}")

SPEED = 150
SPEED_Y = 250
DEAD_ZONE = 0.008

def cursor_loop():
    global prev_x, prev_y
    prev_x, prev_y = 0.0, 0.0
    print("Cursor running. Run stop_cursor.set() to stop.")
    while not stop_cursor.is_set():
        try:
            gx = sum(gyro_buffer_x) / len(gyro_buffer_x)
            gy = sum(gyro_buffer_y) / len(gyro_buffer_y)
            pitch = gx - bias_x
            yaw   = gy - bias_y
            if abs(pitch) < DEAD_ZONE: pitch = 0.0
            if abs(yaw)   < DEAD_ZONE: yaw   = 0.0
            raw_x = yaw * SPEED
            raw_y = pitch * -SPEED_Y
            smooth_x = prev_x + SMOOTHING * (raw_x - prev_x)
            smooth_y = prev_y + SMOOTHING * (raw_y - prev_y)
            prev_x, prev_y = smooth_x, smooth_y
            dx = int(smooth_x)
            dy = int(smooth_y)
            if dx != 0 or dy != 0:
                pyautogui.moveRel(dx, dy, duration=0)
            if check_click():
                pyautogui.click()
                print("CLICK")
        except Exception as e:
            print(f"Error: {e}")
            break
        time.sleep(0.04)

stop_cursor = threading.Event()
cursor_thread = threading.Thread(target=cursor_loop, daemon=True)
cursor_thread.start()

Bias captured: x=0.0215, y=-0.0110
Cursor running. Run stop_cursor.set() to stop.


CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK
CLICK


In [6]:
stop_cursor.set()

In [13]:
from idun_guardian_sdk import FileTypes

client.download_file(recording_id="1781846328087", file_type=FileTypes.EEG)

[INFO] 2026-07-08 15:11:52,771: [API]: File saved: 'eeg_1781846328087.csv'
